In [1]:
from traceback import format_exc

try:
    from src.utils.gerenciador_local_v2 import GerenciadorLocal

    gerenciador_local = GerenciadorLocal(
        nome_sessao="radar-massivo-v9-sprint1",
        exibir_configuracao=False,
        ativar_logs=True,
    )

    spark = gerenciador_local.criar_sessao_spark(
        db2=True,
        driver_memory="16g",
        num_executors=6,
        executor_memory="4g",
        executor_cores=4,
        spark_conf={
            "spark.driver.memoryOverhead": "8g",
            "spark.executor.memoryOverhead": "2g",

            "spark.dynamicAllocation.enabled": "true",
            "spark.dynamicAllocation.minExecutors": "4",
            "spark.dynamicAllocation.initialExecutors": "6",
            "spark.dynamicAllocation.maxExecutors": "8",

            "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
            "spark.kryoserializer.buffer.max": "512m",

            "spark.speculation": "false",
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
            "spark.sql.adaptive.advisoryPartitionSizeInBytes": "64m",
            "spark.sql.adaptive.skewJoin.enabled": "true",
            "spark.sql.adaptive.localShuffleReader.enabled": "true",
            "spark.sql.shuffle.partitions": "320",

            "spark.sql.broadcastTimeout": "8000",
            "spark.executor.heartbeatInterval": "30s",
            "spark.network.timeout": "300s",
            "spark.sql.session.timeZone": "America/Sao_Paulo",
        },
    )

    try:
        widget_cls = __import__("ipywidgets").Widget
        ipython = get_ipython()
        if ipython is not None:
            ipython.display_formatter.formatters["text/plain"].for_type(
                widget_cls,
                lambda *a, **k: None,
            )
    except (ImportError, NameError, AttributeError):
        pass

    print("[RADAR V9] Sessão Spark inicializada.")

except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
48728,application_1794695782786_37104,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.
[RADAR V9] Sessão Spark inicializada.


In [2]:
%run ./src/utils/gerenciador_spark_v2.ipynb
%run ./src/utils/gerenciador_db2_spark_v2.ipynb


Módulo gerenciador_spark carregado com sucesso.

Módulo gerenciador_db2_spark carregado com sucesso (ConectorDb2Spark).

In [5]:
%%spark
# Controles do DDL
NOME_TABELA = "ANL_RADAR_FNC_CLI"
SCHEMA_TABELA = "SBX_A2A13E2"

TABELA_RADAR = f"{SCHEMA_TABELA}.{NOME_TABELA}" if SCHEMA_TABELA else NOME_TABELA


In [6]:
%%spark
DDL_ANA_RADAR_FIN_CLI = f"""CREATE TABLE {TABELA_RADAR} (
    DT_EXEA DATE
        COMMENT 'Data da execução',
    CD_CLI_REF INT
        COMMENT 'MCI avaliado',
    TS_INCL_TRAN_REF TIMESTAMP
        COMMENT 'Última inclusão na janela, obtida com MAX(TS_INCL_TRAN) por CD_CLI',
    FLAG_AGRO STRING
        COMMENT 'Indica se o MCI tem classificação Agro para a decisão de participação',
    FLAG_PESSOA_FISICA STRING
        COMMENT 'Indica se o MCI está identificado como pessoa física',
    FLAG_CONTA_CORRENTE_BB_UNICA STRING
        COMMENT 'Indica se foi identificada uma única conta corrente BB em formato válido para consultar o ciclo',
    DD_INC_MM_CLC_BLC_REF SMALLINT
        COMMENT 'Dia do ciclo cadastrado, nulo quando o cadastro não informa o dia',
    FLAG_CPF_UNICO STRING
        COMMENT 'Indica se foi encontrado exatamente um CPF distinto',
    FLAG_RENDA_PRESUMIDA_DISPONIVEL STRING
        COMMENT 'Indica se a consulta retornou um valor de renda presumida, inclusive zero',
    VL_REN_PRES_REF DECIMAL(17,2)
        COMMENT 'Renda presumida usada no diagnóstico',
    FLAG_MACROPERFIL_VALIDO STRING
        COMMENT 'Indica se o código do macroperfil é 1, 2 ou 3',
    CD_MAC_PRFL_CLI_REF INT
        COMMENT 'Macroperfil usado na pontuação',
    DT_INI_REF DATE
        COMMENT 'Data inicial do período de análise do MCI',
    DT_FIM_REF DATE
        COMMENT 'Data final do período de análise do MCI',
    FLAG_PARTICIPA_RADAR STRING
        COMMENT 'Indica todas as condições de participação foram aprovadas'
)
COMMENT 'Tabela inicial do Radar Financeiro V6, com uma linha por MCI avaliado'
STORED AS PARQUET;
"""

# Recria a tabela vazia a cada execução, usando a sessão Spark existente.
spark.sql(f"DROP TABLE IF EXISTS {TABELA_RADAR}")
spark.sql(DDL_ANA_RADAR_FIN_CLI)


DataFrame[]

In [8]:
%%spark

spark.sql("SHOW TABLES IN SBX_A2A13E2").show(200, truncate=False)

+-----------+-----------------+-----------+
|namespace  |tableName        |isTemporary|
+-----------+-----------------+-----------+
|sbx_a2a13e2|anl_radar_fnc_cli|false      |
|sbx_a2a13e2|anl_cli_fnc      |false      |
|sbx_a2a13e2|ando_anl_cli_fnc |false      |
+-----------+-----------------+-----------+